In [ ]:
# pyright: reportAttributeAccessIssue=false, reportUnusedFunction=false, reportUnusedImport=false, reportWildcardImportFromLibrary=false, reportUndefinedVariable=false, reportArgumentType=false
%load_ext autoreload
%autoreload 2

# core.domclass

>DOM Model with Pydantic and Pandoc Integration
output-file: core.domclass.html
title: core.domclass

This notebook demonstrates a Document Object Model (DOM) using Pydantic for static typing and validation, and integrates Pandoc (via pypandoc) for Markdown processing.
---

In [ ]:
# | default_exp core.domclass

In [ ]:
# | export

In [ ]:
#| export
from fastcore.basics import patch
from fastcore.test import *


In [ ]:
# | export
from collections.abc import Callable
from pathlib import Path
import hashlib
import json

import chromadb
import pypandoc
from chromadb import PersistentClient
from chromadb.api import ClientAPI
from chromadb.config import Settings


In [ ]:
# | hide
import asyncio
import threading
from tempfile import TemporaryDirectory
from types import SimpleNamespace
from unittest.mock import patch as mock_patch
import shutil

In [ ]:
#| export
from ollama import AsyncClient
try:
    from openai import AsyncOpenAI
except ModuleNotFoundError:
    class AsyncOpenAI:
        def __init__(self, *args, **kwargs):
            pass


In [ ]:
# | export
import os
import re
import sys
from pathlib import Path
from typing import ClassVar, Optional, Any, TYPE_CHECKING

project_root = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "pyproject.toml").exists()),
    None,
 )
if project_root is not None and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import jsoncfg
from jsoncfg.config_classes import (
    ConfigJSONObject,
 )
from pydantic import BaseModel, Field
from ribosome.core.dom.summary import (
        summary_node_pair_async, 
        summary_node_async, 
        _summarize_leaf_async,
 )
from ribosome.core.dom.embedding import embed
from ribosome.core.dom.utils import AnalysisStatus, ResponseStatus
from ribosome.core.dom.reorg import reorg_slides, reorg_node

In [ ]:
# | hide
def run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result = {}
    error = {}

    def _runner():
        try:
            result['value'] = asyncio.run(coro)
        except BaseException as exc:
            error['value'] = exc

    thread = threading.Thread(target=_runner, daemon=True)
    thread.start()
    thread.join()
    if 'value' in error:
        raise error['value']
    return result.get('value')

class FakeEmbedResponse:
    def __init__(self, text: str):
        self.embedding = [float(len(text or ''))]
        self.model = 'fake-embed-model'

class FakeCollection:
    def __init__(self):
        self.records = []

    def add(self, ids, metadatas, embeddings, documents):
        self.records.append({
            'ids': ids,
            'metadatas': metadatas,
            'embeddings': embeddings,
            'documents': documents,
        })

class FakeDbClient:
    def __init__(self, collection=None):
        self.collection = collection or FakeCollection()

    def get_or_create_collection(self, name: str):
        return self.collection

def make_dom_fixture(content: str = '# Title\n\nParagraph\n'):
    tmpdir = TemporaryDirectory()
    root = Path(tmpdir.name)
    md_file = root / 'doc.md'
    md_file.write_text(content, encoding='utf-8')
    collection = FakeCollection()
    dom = DOMClass(
        md_file,
        ollama_client=AsyncClient(),
        openai_client=AsyncOpenAI(api_key='test', base_url='https://example.invalid/v1'),
        db_client=PersistentClient(path=str(root / 'db')),
    )
    return tmpdir, root, md_file, dom, collection

## DOM Class with pypandoc Integration

In [ ]:
# | export
class DOMClass(BaseModel):
    """Represent a Markdown document and its derived processing artifacts.

    `DOMClass` keeps the source Markdown together with the JSON artifacts produced
    during parsing, semantic analysis, and embedding, plus the clients and metadata
    needed to transform the document into a structured DOM for downstream retrieval
    workflows.
    """

    model_config = {"arbitrary_types_allowed": True}
    
    # The content of the Markdown document. This can be a string containing Markdown syntax.
    raw_markdown: Optional[str] = Field(None, description="Raw Markdown content")
    # raw json
    raw_json: Optional[str] = Field(None, description="Raw JSON content")
    # The json representation of the Markdown AST.
    ast_json: Optional[str] = Field(None, description="JSON representation of the Markdown AST")
    ast_json_file: Optional[Path] = Field(None, description="Path to the JSON file of the Markdown AST")
    semantics_json: Optional[str] = Field(None, description="Semantics JSON representation of the Markdown AST")
    semantics_json_file: Optional[Path] = Field(None, description="Path to the JSON file of the Markdown AST semantics")
    embed_json: Optional[str] = Field(None, description="Semantics JSON representation with embeddings and meta information of the Markdown AST")
    embed_json_file: Optional[Path] = Field(None, description="Path to the JSON file of the Markdown AST embeddings")
    file_path: Optional[Path] = Field(None, description="Path to the Markdown file")
    root_path: Path = Field(default_factory=Path, description="root path of the markdown document, required to get access to the images")
    table_count: int = Field(0, description="Number of tables in the Markdown document")
    section_count: int = Field(0, description="Number of headers in the Markdown document")
    section_level: list = Field(default_factory=list, description="List of section levels in the Markdown document")
    title:str = Field('', description="Title of the Markdown document, if available")
    ollama_client: AsyncClient = Field(default_factory=AsyncClient, description="AsyncClient instance for embeddings and image analysis (Ollama)")
    ollama_model: str = Field("gemme4:e4b", description="Model name for the Ollama client")
    openai_client: AsyncOpenAI = Field(default_factory=lambda: AsyncOpenAI(api_key=os.environ.get('DASHSCOPE_API_KEY'), base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"), description="AsyncOpenAI instance for text summarization (OpenAI-compatible APIs)")
    openai_model: str = Field("qwen3.6-flash")
    db_client: ClientAPI = Field(default_factory=PersistentClient, description="PersistentClient instance for database interactions")
    analysis_status: AnalysisStatus = Field(
        default_factory=lambda: AnalysisStatus(status=ResponseStatus.PENDING, exception=""),
        description="Analysis status of the Markdown document",
    )


    # Use ClassVar to indicate these are class variables, not instance fields
    TextBlock_Types: ClassVar[set[str]] = {
        "Plain",
        "Para",
        "Figure",
        "LineBlock","CodeBlock","RawBlock","OrderedList","BulletList","DefinitionList",
        "Header","BlockQuote",
        "Table","TableRow", "TableCell"}

    NonTextBlock_Types: ClassVar[set[str]] = {"HorizontalRule", "Div", "Null"}

    Embed_Types: ClassVar[set[str]] = {"Section", "Image", "Table"}

    Block_Types: ClassVar[set[str]] = TextBlock_Types.union(TextBlock_Types)

    Inline_Types: ClassVar[set[str]] = {
        "Str", "Emph", "Strong", "Strikeout", "Superscript", "Subscript",
        "Decimal", "Period",
        "Link", 
        "Image", "Code", "Math", "RawInline", "SoftBreak", "HardBreak", "Span"   
    }
    
    Element_Types: ClassVar[set[str]] = Block_Types | Inline_Types
    leaf_min_len: ClassVar[int] = 100  # Minimum length of text to consider for summarization
    min_len: ClassVar[int] = 300  # Minimum length of the final summary
    lang: ClassVar[str] = "zh"  # Language of the document

    _ARTIFACT_SUFFIXES: ClassVar[dict[str, str]] = {
        "ast_json_file": "_ast.json",
        "semantics_json_file": "_semantics.json",
        "embed_json_file": "_embed.json",
    }

    if TYPE_CHECKING:
        def _document_metadata(self) -> dict[str, str]: ...
        async def _finalize_summary_ast_async(self, _ast_dict: dict, _blocks: list) -> str: ...
        def reorg(self, _action: Optional[Callable] = None) -> str: ...
        @classmethod
        def identity(cls, _obj: Any) -> Any: ...
    

    def __init__(self, md_file_path: Path, ollama_client: Optional[AsyncClient] = None, openai_client: Optional[AsyncOpenAI] = None, db_client: Optional[ClientAPI] = None, **data):
        """Initialize a DOMClass document wrapper for a markdown file.
        
        Args:
            md_file_path: Path to the source markdown file.
            ollama_client: Optional async Ollama client override.
            openai_client: Optional async OpenAI-compatible client override.
            db_client: Optional ChromaDB client override.
            **data: Additional model fields passed to `BaseModel`.
        """
        # Set defaults for data if not provided
        data.setdefault('file_path', Path(md_file_path))
        data.setdefault('root_path', Path(md_file_path).parent)
        data.setdefault('title', Path(md_file_path).stem)
        data.setdefault('ollama_client', ollama_client or AsyncClient())
        data.setdefault('ollama_model', "gemme4:e4b")
        data.setdefault('openai_client', openai_client or AsyncOpenAI(api_key=os.environ.get('DASHSCOPE_API_KEY'), base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"))
        data.setdefault('openai_model', "qwen3.6-flash")
        data.setdefault('db_client', db_client or PersistentClient())
        data.setdefault('ast_json_file', None)
        data.setdefault('embed_json_file', None)
        data.setdefault('semantics_json_file', None)

        super().__init__(**data)

    def _artifact_path(self, suffix: str) -> Path:
        assert self.file_path is not None, "file_path is required to build artifact paths"
        return self.file_path.parent / f"{self.file_path.stem}{suffix}"

    def _sync_artifact_fields(self) -> None:
        for attr_name, suffix in self._ARTIFACT_SUFFIXES.items():
            setattr(self, attr_name, self._artifact_path(suffix))

    def _load_artifact(self, path_attr: str, content_attr: str) -> None:
        path = getattr(self, path_attr)
        if path and path.exists():
            setattr(self, content_attr, path.read_text(encoding="utf-8"))

    def _reset_document_state(self) -> None:
        self.raw_markdown = None
        self.raw_json = None
        self.ast_json = None
        self.semantics_json = None
        self.embed_json = None
        self.ast_json_file = None
        self.semantics_json_file = None
        self.embed_json_file = None

    def _require_action(self, action: Optional[Callable]) -> Callable:
        return action or self.__class__.identity

    def _render_markdown(self, target: str) -> str | None:
        if not self.raw_markdown:
            return None
        return pypandoc.convert_text(self.raw_markdown, target, "md")

    def _dump_json(self, value: dict | list) -> str:
        return json.dumps(value, ensure_ascii=False).encode("utf-8").decode("utf-8")


In [ ]:
# | hide
def test_domclass_core_helpers():
    tmpdir, root, md_file, dom, _ = make_dom_fixture()
    test_eq(dom.file_path, md_file)
    test_eq(dom.root_path, root)
    test_eq(dom.title, 'doc')
    test_eq(dom.analysis_status.status, ResponseStatus.PENDING)
    test_eq(dom._artifact_path('_ast.json'), root / 'doc_ast.json')
    dom._sync_artifact_fields()
    test_eq(dom.ast_json_file, root / 'doc_ast.json')
    dom.ast_json_file.write_text('{"blocks":[1]}', encoding='utf-8')
    dom._load_artifact('ast_json_file', 'ast_json')
    test_eq(dom.ast_json, '{"blocks":[1]}')
    dom.raw_markdown = 'Paragraph'
    with mock_patch.object(pypandoc, 'convert_text', return_value='<p>Paragraph</p>'):
        test_eq(dom._render_markdown('html'), '<p>Paragraph</p>')
    test_eq(dom._dump_json({'text': '你好'}), '{"text": "你好"}')
    def identity_action(value):
        return value
    test_eq(dom._require_action(identity_action), identity_action)
    dom.raw_json = 'filled'
    dom.ast_json = 'filled'
    dom.semantics_json = 'filled'
    dom.embed_json = 'filled'
    dom._reset_document_state()
    test_eq(dom.raw_markdown, None)
    test_eq(dom.raw_json, None)
    test_eq(dom.ast_json, None)
    test_eq(dom.semantics_json, None)
    test_eq(dom.embed_json, None)
    tmpdir.cleanup()


test_domclass_core_helpers()

In [ ]:
# | export
def _identity_bootstrap(_obj: Any) -> Any:
    return _obj

if 'DOMClass' in globals():
    DOMClass.identity = staticmethod(_identity_bootstrap)

In [ ]:
# | export
@patch
def _document_metadata(self: DOMClass) -> dict[str, str]:
    """Get the document metadata.

    Args:
        self (DOMClass): The DOMClass instance.

    Returns:
        dict[str, str]: The document metadata.
    """
    if self.title is None:
        self.title = str(self.root_path)
    return {
        'title': self.title,
        'file_path': str(self.file_path),
    }


In [ ]:
# | hide
def test_document_metadata_uses_title_and_path():
    tmpdir, _, md_file, dom, _ = make_dom_fixture()
    test_eq(dom._document_metadata(), {'title': 'doc', 'file_path': str(md_file)})
    tmpdir.cleanup()


test_document_metadata_uses_title_and_path()

In [ ]:
# | export
@patch
async def _finalize_summary_ast_async(self: DOMClass, ast_dict: dict, blocks: list) -> str:
    """Finalize the summary of the AST asynchronously.

    Args:
        self (DOMClass): The DOMClass instance.
        ast_dict (dict): The AST dictionary.
        blocks (list): The list of blocks to be summarized.

    Returns:
        str: The serialized summary of the AST.
    """
    ast_dict.update(self._document_metadata())
    dict_summary = [b['s'] for b in blocks if isinstance(b, dict) and 's' in b]
    dict_summary = [ast_dict['title']] + dict_summary
    ast_dict['summary'] = "" if not ast_dict['blocks'] or dict_summary == [] else await _summarize_leaf_async(
        " ".join(dict_summary), 
        leaf_min_len=self.leaf_min_len,
        client=self.ollama_client,
        model=self.ollama_model, 
        min_len=self.min_len,
        lang="zh")
    serialized = self._dump_json(ast_dict)
    self.ast_json = serialized
    return serialized


In [ ]:
# | hide
def test_finalize_summary_ast_async_updates_summary_and_cache():
    tmpdir, _, _, dom, _ = make_dom_fixture()
    async def fake_summary(node, leaf_min_len, client, model, min_len, lang):
        return f'S<{node}>'
    with mock_patch.dict(globals(), {'_summarize_leaf_async': fake_summary}):
        serialized = run_async(dom._finalize_summary_ast_async({'blocks': [{'t': 'Para'}]}, [{'s': 'alpha'}]))
    result = json.loads(serialized)
    test_eq(result['title'], 'doc')
    test_eq(result['summary'], 'S<doc alpha>')
    test_eq(dom.ast_json, serialized)
    tmpdir.cleanup()


test_finalize_summary_ast_async_updates_summary_and_cache()

In [ ]:
# | export
@patch
def setup(self: DOMClass):
    """Load markdown content and initialize cached AST, semantics, and embedding artifacts.
    
    Args:
        None.
    """
    content = self.file_path.read_text(encoding="utf-8")  # type: ignore
    if not content:
        self._reset_document_state()
        return

    self.raw_markdown = content
    self.raw_json = pypandoc.convert_text(self.raw_markdown, "json", "md")
    self._sync_artifact_fields()

    if self.ast_json_file and self.ast_json_file.exists():
        self.ast_json = self.ast_json_file.read_text(encoding="utf-8")
    else:
        print(f"--- IGNORE --- {self.file_path}")
        slide_splitter = r"(^<!--\s*Slide number:\s*\d+\s*-->$)"
        has_slides = re.search(slide_splitter, self.raw_markdown, flags=(re.MULTILINE | re.IGNORECASE))
        self.ast_json = reorg_slides(raw_json=self.raw_json,
                                    raw_markdown=self.raw_markdown,
                                    slide_splitter=slide_splitter) if has_slides else self.reorg()
        if self.ast_json_file and self.ast_json is not None:
            self.ast_json_file.write_text(self.ast_json, encoding="utf-8")

    self._load_artifact("semantics_json_file", "semantics_json")
    self._load_artifact("embed_json_file", "embed_json")

In [ ]:
# | hide
def test_setup_loads_cached_artifacts_and_resets_empty_docs():
    tmpdir, root, _, dom, _ = make_dom_fixture('# Title\n')
    (root / 'doc_ast.json').write_text('{"blocks": ["cached"]}', encoding='utf-8')
    (root / 'doc_semantics.json').write_text('{"summary": "semantic"}', encoding='utf-8')
    (root / 'doc_embed.json').write_text('{"summary": "embed"}', encoding='utf-8')
    with mock_patch.object(pypandoc, 'convert_text', return_value='{"blocks": ["raw"]}'):
        dom.setup()
    test_eq(dom.raw_markdown, '# Title\n')
    test_eq(dom.raw_json, '{"blocks": ["raw"]}')
    test_eq(dom.ast_json, '{"blocks": ["cached"]}')
    test_eq(dom.semantics_json, '{"summary": "semantic"}')
    test_eq(dom.embed_json, '{"summary": "embed"}')
    tmpdir.cleanup()

    tmpdir, _, _, dom, _ = make_dom_fixture('')
    dom.raw_markdown = 'set'
    dom.raw_json = 'set'
    dom.setup()
    test_eq(dom.raw_markdown, None)
    test_eq(dom.raw_json, None)
    tmpdir.cleanup()


test_setup_loads_cached_artifacts_and_resets_empty_docs()

In [ ]:
# | export
@patch
def to_markdown(self: DOMClass) -> str | None:
    """Return the raw markdown content for the current document.
    
    Args:
        None.
    
    Returns:
        str | None: Stored markdown text, if available.
    """
    return self.raw_markdown


In [ ]:
# | hide
def test_to_markdown_returns_raw_markdown():
    tmpdir, _, _, dom, _ = make_dom_fixture()
    dom.raw_markdown = 'Paragraph'
    test_eq(dom.to_markdown(), 'Paragraph')
    tmpdir.cleanup()


test_to_markdown_returns_raw_markdown()

In [ ]:
# | export
@patch
def to_html(self: DOMClass) -> str | None:
    """Render the current markdown content to HTML.
    
    Args:
        None.
    
    Returns:
        str | None: Rendered HTML, or `None` when no markdown is loaded.
    """
    return self._render_markdown("html")


In [ ]:
# | hide
def test_to_html_renders_markdown():
    tmpdir, _, _, dom, _ = make_dom_fixture()
    dom.raw_markdown = 'Paragraph'
    with mock_patch.object(pypandoc, 'convert_text', return_value='<p>Paragraph</p>'):
        test_eq(dom.to_html(), '<p>Paragraph</p>')
    tmpdir.cleanup()


test_to_html_renders_markdown()

In [ ]:
# | export
@patch
def to_latex(self: DOMClass) -> str | None:
    """Render the current markdown content to LaTeX.
    
    Args:
        None.
    
    Returns:
        str | None: Rendered LaTeX, or `None` when no markdown is loaded.
    """
    return self._render_markdown("latex")


In [ ]:
# | hide
def test_to_latex_renders_markdown():
    tmpdir, _, _, dom, _ = make_dom_fixture()
    dom.raw_markdown = 'Paragraph'
    with mock_patch.object(pypandoc, 'convert_text', return_value='\\paragraph{Paragraph}'):
        test_eq(dom.to_latex(), '\\paragraph{Paragraph}')
    tmpdir.cleanup()


test_to_latex_renders_markdown()

In [ ]:
# | export
@patch
def to_json(self: DOMClass) -> str | None:
    """Return the serialized Pandoc AST for the current document.
    
    Args:
        None.
    
    Returns:
        str | None: Stored AST JSON, if available.
    """
    return self.raw_json


In [ ]:
# | hide
def test_to_json_returns_raw_json():
    tmpdir, _, _, dom, _ = make_dom_fixture()
    dom.raw_json = '{"blocks": []}'
    test_eq(dom.to_json(), '{"blocks": []}')
    tmpdir.cleanup()


test_to_json_returns_raw_json()

In [ ]:
# | export
@patch
async def embed_doc(self: DOMClass, action: Optional[Callable] = None, db_path: Optional[Path] = Path("../db")) -> None:
    """Embed summarized semantic nodes and persist the embedding JSON output.
    
    Args:
        action: Optional transform applied to each node before traversal.
        db_path: Path to the persistent Chroma database directory.
    
    Returns:
        None
    """
    if not self.semantics_json:
        raise ValueError("semantics_json content is empty. Cannot embed the content.")
    action = self._require_action(action)

    if self.embed_json:
        # If embed_json is already set, we don't need to walk the AST again
        print("embed_json is already set. Skipping walk.")
        return

    try:
        self.db_client = PersistentClient(path=str(db_path), settings=Settings(allow_reset=True))  # type: ignore
        # ephemeral_db_client = chromadb.EphemeralClient()  # type: ignore     
        # assert ephemeral_db_client, "Failed to create ephemeral ChromaDB client."
    except Exception as e:
        print(f"Failed to create PersistentClient: {e}")
    # Create or get the collection in the database 'mitochondria'
    collection = self.db_client.get_or_create_collection(name="mitochondria")  # defined in the closure for embed_node
    # collection = ephemeral_db_client.get_or_create_collection(name="mitochondria")
    assert collection, "Failed to create or get the 'mitochondria' collection in the database."        


    assert self.file_path, "file path is not set."
    self.embed_json = await embed(
        semantics_json=self.semantics_json,
        file_path=self.file_path,
        embed_types=self.Embed_Types,
        db_client=self.db_client,
        collection=collection,
        llm_client=self.ollama_client,
        model=self.ollama_model,
        action=action,
    )

    embed_json_file = self.file_path.parent / f"{self.file_path.stem}_embed.json"
    assert isinstance(self.embed_json, str), "embed_json should be a string."
    with open(embed_json_file, "w", encoding="utf-8") as f:
        f.write(self.embed_json)

In [ ]:
# | hide
def test_embed_doc_embeds_semantics_and_writes_artifact():
    tmpdir, root, _, dom, collection = make_dom_fixture()
    dom.semantics_json = '{"summary": "semantic"}'
    captured = {}
    def fake_persistent_client(path, settings):
        captured['path'] = path
        captured['settings'] = settings
        return FakeDbClient(collection)
    async def fake_embed(**kwargs):
        captured['embed_kwargs'] = kwargs
        return '{"embedded": true}'
    with mock_patch.dict(globals(), {'PersistentClient': fake_persistent_client, 'Settings': lambda allow_reset=True: SimpleNamespace(allow_reset=allow_reset), 'embed': fake_embed}):
        run_async(dom.embed_doc())
    test_eq(dom.embed_json, '{"embedded": true}')
    test_eq(captured['embed_kwargs']['semantics_json'], '{"summary": "semantic"}')
    test_eq((root / 'doc_embed.json').read_text(encoding='utf-8'), '{"embedded": true}')
    tmpdir.cleanup()

    tmpdir, _, _, dom, _ = make_dom_fixture()
    test_fail(lambda: run_async(dom.embed_doc()), contains='semantics_json content is empty')
    tmpdir.cleanup()


test_embed_doc_embeds_semantics_and_writes_artifact()

In [ ]:
# | export
@patch
def reorg(self: DOMClass, action: Optional[Callable] = None) -> str:
    """Convert header-based AST blocks into nested section structures.
    
    Args:
        action: Optional transform applied to each visited node.
    
    Returns:
        str: Reorganized AST serialized as JSON.
    """
    if not self.raw_json:
        raise ValueError("raw_json content is empty. Cannot walk the AST.")
    action = self._require_action(action)

    ast = json.loads(self.raw_json)

    ast = reorg_node(
        node=ast,
        section_level=self.section_level,
        action=self.identity
    )
    return self._dump_json(ast)

In [ ]:
# | hide
def test_reorg_uses_reorg_node_and_requires_raw_json():
    tmpdir, _, _, dom, _ = make_dom_fixture()
    dom.raw_json = '{"blocks": []}'
    def fake_reorg_node(node, section_level, action):
        node['token'] = action('ok')
        node['blocks'] = [{'t': 'Section', 'c': []}]
        return node
    with mock_patch.dict(globals(), {'reorg_node': fake_reorg_node}):
        result = json.loads(dom.reorg())
    test_eq(result['token'], 'ok')
    test_eq(result['blocks'][0]['t'], 'Section')
    tmpdir.cleanup()

    tmpdir, _, _, dom, _ = make_dom_fixture()
    dom.raw_json = None
    test_fail(lambda: dom.reorg(), contains='raw_json content is empty')
    tmpdir.cleanup()


test_reorg_uses_reorg_node_and_requires_raw_json()

In [ ]:
# | export
@patch
async def textualize(self: DOMClass, action: Optional[Callable] = None) -> None:
    """Summarize AST nodes into semantic text fields for downstream embedding.
    
    Args:
        action: Optional transform applied to each visited node.
    """
    if not self.ast_json:
        raise ValueError("raw_json content is empty. Cannot walk the AST and summarize.")
    action = self._require_action(action)


    config_ast = jsoncfg.load_config(str(self.ast_json_file))
    ast_dict = json.loads(self.ast_json)
    if not isinstance(config_ast, ConfigJSONObject):
        raise ValueError(f"AST should be a dictionary, got {type(config_ast)}")

    config_blocks = config_ast['blocks']
    blocks = ast_dict['blocks']
    assert isinstance(self.file_path, Path), "self.file_path should be a Path object"
    _, blocks = await summary_node_pair_async(
        config_node=config_blocks, 
        node=blocks, 
        root_path=
        self.root_path,client=self.ollama_client,
        model="gemma4:e4b",
        file_path=self.file_path,
        table_count=self.table_count,
        section_count=self.section_count
        )
    ast_dict['blocks'] = blocks
    assert isinstance(blocks,list)
    await self._finalize_summary_ast_async(ast_dict, blocks)
    return

    # # Run the summary_nodewline_main function asynchronously
    # # asyncio.run(summary_nodewline_main(action))
    # await summary_nodewline_main(action)
    # return 


In [ ]:
# | hide
def test_textualize_updates_blocks_and_document_summary():
    tmpdir, root, _, dom, _ = make_dom_fixture()
    dom.ast_json_file = root / 'doc_ast.json'
    dom.ast_json = '{"blocks": [{"t": "Para", "c": []}]}'
    dom.ast_json_file.write_text(dom.ast_json, encoding='utf-8')
    async def fake_summary_pair(config_node, node, root_path, client, model, file_path, table_count, section_count):
        return config_node, [{'t': 'Para', 'c': [], 's': 'section summary'}]
    async def fake_summary(node, leaf_min_len, client, model, min_len, lang):
        return f'S<{node}>'
    with mock_patch.dict(globals(), {'summary_node_pair_async': fake_summary_pair, '_summarize_leaf_async': fake_summary}):
        run_async(dom.textualize())
    result = json.loads(dom.ast_json)
    test_eq(result['blocks'][0]['s'], 'section summary')
    test_eq(result['summary'], 'S<doc section summary>')
    tmpdir.cleanup()

    tmpdir, _, _, dom, _ = make_dom_fixture()
    dom.ast_json = None
    test_fail(lambda: run_async(dom.textualize()), contains='raw_json content is empty')
    tmpdir.cleanup()


test_textualize_updates_blocks_and_document_summary()

In [ ]:
# | export
@patch
def identity(self: DOMClass, obj: Any) -> Any:
    """Return an object unchanged during traversal hooks.
    
    Args:
        obj: Object to return as-is.
    
    Returns:
        Any: The original object.
    """
    return obj


In [ ]:
# | hide
def test_identity_returns_original_object():
    tmpdir, _, _, dom, _ = make_dom_fixture()
    payload = {'x': 1}
    test_eq(dom.identity(payload), payload)
    test_eq(dom._require_action(None), DOMClass.identity)
    tmpdir.cleanup()


test_identity_returns_original_object()

In [ ]:
# | export
@patch
async def summary_nodewline_main(self: DOMClass,action: Optional[Callable] = None) -> None:
    """
    Main function to walk the AST and summarize nodes.
    This function will be called by the walk method.
    """

    if not self.ast_json:
        raise ValueError("raw_json content is empty. Cannot walk the AST and summarize.")
    if action is None:
        action = self.__class__.identity
    config_ast = jsoncfg.load_config(str(self.ast_json_file))
    ast_dict = json.loads(self.ast_json)
    assert isinstance(config_ast, ConfigJSONObject), f"AST should be a dictionary, got {type(config_ast)}"
    config_blocks = config_ast['blocks']
    blocks = ast_dict['blocks']
    assert self.ast_json_file, "ast_json_file not ready!"
    _ , blocks = await summary_node_pair_async(
        config_node=config_blocks, 
        node=blocks, 
        root_path=self.root_path,
        file_path=self.ast_json_file,
        client=self.ollama_client,
        model=self.ollama_model,
        table_count=self.table_count,
        section_count=self.section_count,
        action=self.identity,
        leaf_min_len=self.leaf_min_len,
        min_len=self.min_len,
        lang=self.lang)
    ast_dict['blocks'] = blocks
    if ast_dict.get('title') is None:
        # If the title is not set, use the file name as the title
        if self.title is None:
            self.title = str(self.root_path)
    ast_dict['title'] = self.title  # Add the title to the AST
    ast_dict['file_path'] = str(self.file_path)  # Add the file path to the AST
    assert isinstance(blocks, list), f"Expected a list of blocks, got {type(blocks)}"
    dict_summary = [b['s'] for b in blocks if isinstance(b, dict) and 's' in b]
    dict_summary = [ast_dict['title']] + dict_summary  # Add the title to the summary list
    # the summary of the document from the summaries in the list of blocks
    if not ast_dict['blocks'] or dict_summary == []:
        # If the blocks are empty, set the summary to an empty string
        ast_dict['summary'] = ""
        # If the summary is empty, set the AST JSON to an empty string
        self.ast_json = json.dumps(ast_dict, ensure_ascii=False).encode("utf-8").decode("utf-8")
    else:
        # If the blocks are not empty, set the summary to the concatenated summaries
        doc_summary = await _summarize_leaf_async(
            node=" ".join(dict_summary),
            leaf_min_len=self.leaf_min_len,
            min_len=self.min_len,
            client=self.ollama_client,
            model=self.ollama_model,
            lang=self.lang
            )
        ast_dict['summary'] = doc_summary
        # Convert the summarized AST back to JSON
        self.ast_json = json.dumps(ast_dict, ensure_ascii=False).encode("utf-8").decode("utf-8")

In [ ]:
# | hide
def test_summary_nodewline_main_updates_ast_json():
    tmpdir, root, _, dom, _ = make_dom_fixture()
    dom.ast_json_file = root / 'doc_ast.json'
    dom.ast_json = '{"blocks": [{"t": "Para", "c": []}]}'
    dom.ast_json_file.write_text(dom.ast_json, encoding='utf-8')
    async def fake_summary_pair(config_node, node, root_path, file_path, client, model, table_count, section_count, action, leaf_min_len, min_len, lang):
        return config_node, [{'t': 'Para', 'c': [], 's': 'section summary'}]
    async def fake_summary(node, leaf_min_len, min_len, client, model, lang):
        return f'S<{node}>'
    with mock_patch.dict(globals(), {'summary_node_pair_async': fake_summary_pair, '_summarize_leaf_async': fake_summary}):
        run_async(dom.summary_nodewline_main())
    result = json.loads(dom.ast_json)
    test_eq(result['blocks'][0]['s'], 'section summary')
    test_eq(result['summary'], 'S<doc section summary>')
    tmpdir.cleanup()


test_summary_nodewline_main_updates_ast_json()

In [ ]:
# | export
@patch
async def summary_node_main(self: DOMClass, action: Optional[Callable] = None) -> None:
    """
    Main function to walk the AST and summarize nodes.
    This function will be called by the walk method.
    """

    if not self.ast_json:
        raise ValueError("raw_json content is empty. Cannot walk the AST and summarize.")
    if action is None:
        action = self.__class__.identity
    ast = json.loads(self.ast_json)
    blocks = ast.get("blocks", [])
    assert isinstance(self.file_path, Path), "self.file_path should be a Path object"
    ast['blocks'] = await summary_node_async(
        node=blocks,
        root_path=self.root_path,
        file_path=self.file_path,
        client=self.ollama_client,
        model=self.ollama_model,
        table_count=self.table_count,
        section_count=self.section_count,
        action=self.identity,
        leaf_min_len=self.leaf_min_len,
        min_len=self.min_len,
        lang=self.lang
        )
    if ast.get('title') is None:
        # If the title is not set, use the file name as the title
        if self.title is None:
            self.title = str(self.root_path)
    ast['title'] = self.title  # Add the title to the AST
    ast['file_path'] = str(self.file_path)  # Add the file path to the AST
    assert isinstance(blocks, list), f"Expected a list of blocks, got {type(blocks)}"
    dict_summary = [b['s'] for b in blocks if isinstance(b, dict) and 's' in b]
    dict_summary = [ast['title']] + dict_summary  # Add the title to the summary list
    # the summary of the document from the summaries in the list of blocks
    if not ast['blocks'] or dict_summary == []:
        # If the blocks are empty, set the summary to an empty string
        ast['summary'] = ""
        # If the summary is empty, set the AST JSON to an empty string
        self.ast_json = json.dumps(ast, ensure_ascii=False).encode("utf-8").decode("utf-8")
    else:
        # If the blocks are not empty, set the summary to the concatenated summaries
        doc_summary = await _summarize_leaf_async(
            node=" ".join(dict_summary),
            leaf_min_len=self.leaf_min_len,
            client=self.ollama_client,
            model=self.ollama_model,
            min_len=self.min_len,
            lang=self.lang
        )
        ast['summary'] = doc_summary
        # Convert the summarized AST back to JSON
        self.ast_json = json.dumps(ast, ensure_ascii=False).encode("utf-8").decode("utf-8")

In [ ]:
# | hide
def test_summary_node_main_updates_ast_json():
    tmpdir, _, _, dom, _ = make_dom_fixture()
    dom.ast_json = '{"blocks": [{"t": "Para", "c": []}]}'
    async def fake_summary_node_async(node, root_path, file_path, client, model, table_count, section_count, action, leaf_min_len, min_len, lang):
        node[0]['s'] = 'section summary'
        return node
    async def fake_summary(node, leaf_min_len, client, model, min_len, lang):
        return f'S<{node}>'
    with mock_patch.dict(globals(), {'summary_node_async': fake_summary_node_async, '_summarize_leaf_async': fake_summary}):
        run_async(dom.summary_node_main())
    result = json.loads(dom.ast_json)
    test_eq(result['blocks'][0]['s'], 'section summary')
    test_eq(result['summary'], 'S<doc section summary>')
    tmpdir.cleanup()


test_summary_node_main_updates_ast_json()

# DomClass Function Tests

## Reorg

In [ ]:
# | hide
def test_reorg_with_sn024002_full_md():
    md_src = Path('res/SN024002/SN024002_full.md')
    tmpdir = TemporaryDirectory()
    root = Path(tmpdir.name)
    md_file = root / md_src.name
    shutil.copy(md_src, md_file)

    dom = DOMClass(md_file)
    dom.raw_markdown = md_file.read_text(encoding='utf-8')
    dom.raw_json = pypandoc.convert_text(dom.raw_markdown, 'json', 'md')

    result_json = dom.reorg()

    test_ne(result_json, None)
    result = json.loads(result_json)
    test_eq('blocks' in result, True)
    blocks = result['blocks']
    test_ne(blocks, [])
    # Headers are reorganised into Section nodes; no bare Header nodes remain at top level
    block_types = [b.get('t') for b in blocks if isinstance(b, dict)]
    test_eq('Section' in block_types, True)
    test_eq(block_types.count('Header'), 0)

    tmpdir.cleanup()


test_reorg_with_sn024002_full_md()

## Summary

In [ ]:
# | hide
def test_textualize_with_sn024002_full_md():
    md_src = Path('res/SN024002/SN024002_full.md')
    if not md_src.exists():
        md_src = Path('../res/SN024002/SN024002_full.md')
    tmpdir = TemporaryDirectory()
    root = Path(tmpdir.name)
    md_file = root / md_src.name
    shutil.copy(md_src, md_file)

    dom = DOMClass(md_file)
    dom.setup()

    test_ne(dom.ast_json, None)
    test_ne(dom.ast_json_file, None)
    ast_pre = json.loads(dom.ast_json)
    test_ne(ast_pre.get('blocks'), [])

    async def fake_summary_pair(config_node, node, root_path, client, model, file_path, table_count, section_count):
        return config_node, [dict(b, s='slide summary') if isinstance(b, dict) else b for b in node]

    async def fake_leaf_summary(node, leaf_min_len, client, model, min_len, lang):
        return 'document summary'

    with mock_patch.dict(globals(), {
        'summary_node_pair_async': fake_summary_pair,
        '_summarize_leaf_async': fake_leaf_summary,
    }):
        run_async(dom.textualize())

    result = json.loads(dom.ast_json)
    test_eq(result['title'], md_src.stem)
    test_eq(result['file_path'], str(md_file))
    test_eq(result['summary'], 'document summary')
    for block in result['blocks']:
        if isinstance(block, dict):
            test_eq('s' in block, True)

    tmpdir.cleanup()


test_textualize_with_sn024002_full_md()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()